In [1]:
!pip install tensorflow opencv-python matplotlib

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import cv2
import os
import random
import numpy as np
import matplotlib.pyplot as plt

In [2]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras import layers

In [3]:
pos_p = os.path.join('data', 'positive')
neg_p = os.path.join('data', 'negative')
anc_p = os.path.join('data', 'anchor')

In [7]:
os.makedirs(pos_p)
os.makedirs(neg_p)
os.makedirs(anc_p)

FileExistsError: [WinError 183] Cannot create a file when that file already exists: 'data\\positive'

In [5]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("jessicali9530/lfw-dataset")

print("Path to dataset files:", path)

100%|██████████████████████████████████████████████████████████████████████████████| 112M/112M [01:55<00:00, 1.02MB/s]

Extracting files...


Path to dataset files: C:\Users\abhay\.cache\kagglehub\datasets\jessicali9530\lfw-dataset\versions\4


In [4]:
for dir in os.listdir('lfw'):
    for file in os.listdir(os.path.join('lfw', dir)):
        ex = os.path.join('lfw', dir, file)
        new = os.path.join(neg_p, file)
        os.replace(ex, new)

In [5]:
# for generating unique id
import uuid

In [36]:
cap = cv2.VideoCapture(0)
while cap.isOpened():
    ret, frame = cap.read()

    # resizing the frame to 250 x 250
    frame = frame[170:170+250, 210:210+250, :]

    # collecting anchors
    if cv2.waitKey(1) & 0XFF == ord('a'):
        imgname = os.path.join(anc_p, '{}.jpg'.format(uuid.uuid1()))
        cv2.imwrite(imgname, frame)

    # collecting positives
    if cv2.waitKey(1) & 0XFF == ord('p'):
        imgname = os.path.join(pos_p, '{}.jpg'.format(uuid.uuid1()))
        cv2.imwrite(imgname, frame)

    cv2.imshow("Image Collection", frame)
    if cv2.waitKey(1) & 0XFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

In [10]:
pos_p

'data\\positive'

In [11]:
uuid.uuid1()

UUID('6b5be89b-154f-11f0-a34e-387a0ed65730')

## pipeline

In [6]:
anchor = tf.data.Dataset.list_files(anc_p + '\*.jpg').take(300)
positive = tf.data.Dataset.list_files(pos_p + '\*.jpg').take(300)
negative = tf.data.Dataset.list_files(neg_p + '\*.jpg').take(300)

In [7]:
def preprocess(file_path):
    byte_img = tf.io.read_file(file_path)
    img = tf.io.decode_jpeg(byte_img)
    img = tf.image.resize(img, (100, 100))
    img = img / 255.0
    return img

In [8]:
positives = tf.data.Dataset.zip((anchor, positive, tf.data.Dataset.from_tensor_slices(tf.ones(len(anchor)))))
negatives = tf.data.Dataset.zip((anchor, negative, tf.data.Dataset.from_tensor_slices(tf.ones(len(anchor)))))
data = positives.concatenate(negatives)

In [9]:
def pre(input_img, val_img, label):
    # Used tf.py_function instead of directly calling preprocess for TensorFlow compatibility
    input_img = tf.py_function(preprocess, [input_img], tf.float32)
    val_img = tf.py_function(preprocess, [val_img], tf.float32)
    return input_img, val_img, label

In [10]:
data = data.map(pre)
data = data.cache()
data = data.shuffle(buffer_size=1024)

In [11]:
train_data = data.take(round(len(data)*0.7))
train_data = train_data.batch(16)
train_data = train_data.prefetch(8)

In [12]:
test_data = data.skip(round(len(data)*0.7))
test_data = test_data.take(round(len(data)*0.3))
test_data = test_data.batch(16)
test_data = test_data.prefetch(8)

## Model engineering

In [13]:
def make_embedding():
    inp = layers.Input(shape=(100,100,3), name='input_image')
    
    c1 = layers.Conv2D(64, (10, 10), activation='relu')(inp)
    m1 = layers.MaxPooling2D(64, (2,2), padding='same')(c1)
    
    c2 = layers.Conv2D(128, (7, 7), activation='relu')(m1)
    m2 = layers.MaxPooling2D(64, (2,2), padding='same')(c2)
    
    c3 = layers.Conv2D(128, (4, 4), activation='relu')(m2)
    m3 = layers.MaxPooling2D(64, (2,2), padding='same')(c3)
    
    c4 = layers.Conv2D(256, (4, 4), activation='relu')(m3)

    f1 = layers.Flatten()(c4)
    d1 = layers.Dense(4096, activation='sigmoid')(f1)
    
    return Model(inputs=[inp], outputs=d1, name='embedding') 

In [14]:
model = make_embedding()

In [15]:
model.summary()

Model: "embedding"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_image (InputLayer)             │ (None, 100, 100, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d (Conv2D)                      │ (None, 91, 91, 64)          │          19,264 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 46, 46, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 40, 40, 128)         │         401,536 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 20, 20, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 17, 17, 128)         │         262,272 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 9, 9, 128)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_3 (Conv2D)                    │ (None, 6, 6, 256)           │         524,544 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 9216)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 4096)                │      37,752,832 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 38,960,448 (148.62 MB)

 Trainable params: 38,960,448 (148.62 MB)

 Non-trainable params: 0 (0.00 B)

In [28]:
class L1Dist(layers.Layer):
    def __init__(self, **kwargs):
        super().__init__()

    def call(self, inputs):
        input, validation = inputs
        return tf.math.abs(input - validation)

    def get_config(self):
        return super().get_config()

In [17]:
l1 = L1Dist() 

In [18]:
def make_siamese_model():
    input_image = layers.Input(name='input_img', shape=(100, 100, 3))
    validation_image = layers.Input(name='validation_img', shape=(100, 100, 3))

    input_embedding = model(input_image)             # ✅ NOT wrapped in list
    validation_embedding = model(validation_image)   # ✅

    siamese_layer = L1Dist()
    siamese_layer._name = 'distance'
    distances = siamese_layer([input_embedding, validation_embedding])  # ✅ wrapped once

    classifier = layers.Dense(1, activation='sigmoid')(distances)

    return Model(inputs=[input_image, validation_image], outputs=classifier, name='SiameseNetwork')


In [22]:
siamese_model = make_siamese_model()

In [19]:
adam = tf.keras.optimizers.Adam(1e-4)

In [20]:
binary_cross_loss = tf.losses.BinaryCrossentropy()

In [23]:
checkpoint_dir = './training_checkpoints'
cp_prefix = os.path.join(checkpoint_dir, 'ckpt')
cp = tf.train.Checkpoint(optimizer=adam, model=siamese_model)

In [23]:
@tf.function
def train_step(batch):
    with tf.GradientTape() as tape:
        x = batch[:2]
        y = batch[2]

        yhat = siamese_model(x, training=True)
        loss = binary_cross_loss(y, yhat)

    print(loss)
    grad = tape.gradient(loss, siamese_model.trainable_variables)
    adam.apply_gradients(zip(grad, siamese_model.trainable_variables))
    return loss

In [24]:
def train(data, epochs):
    for epoch in range(1, epochs + 1):
        print('\nEpoch {}/{}'.format(epoch, epochs))
        progbar = tf.keras.utils.Progbar(len(data))

        total_loss = 0  # Initialize total loss
        num_batches = 0  # Track batch count

        for idx, batch in enumerate(data):
            batch_loss = train_step(batch)  # Compute loss for the batch
            total_loss += batch_loss.numpy()  # Accumulate loss
            num_batches += 1

            progbar.update(idx + 1)

        avg_loss = total_loss/num_batches
        print("Epoch {} Loss: {:.4f}".format(epoch, avg_loss))  # Print final loss for the epoch

        if epoch % 10 == 0:
            cp.save(file_prefix=cp_prefix)  

In [25]:
epochs = 50

In [74]:
train(train_data, epochs)


Epoch 1/50
7/7 [==============================] - 95s 13s/step
Epoch 1 Loss: 0.0041

Epoch 2/50
7/7 [==============================] - 97s 14s/step
Epoch 2 Loss: 0.0044

Epoch 3/50
7/7 [==============================] - 100s 14s/step
Epoch 3 Loss: 0.0013

Epoch 4/50
7/7 [==============================] - 100s 14s/step
Epoch 4 Loss: 0.0012

Epoch 5/50
7/7 [==============================] - 100s 14s/step
Epoch 5 Loss: 0.0006

Epoch 6/50
7/7 [==============================] - 104s 15s/step
Epoch 6 Loss: 0.0004

Epoch 7/50
7/7 [==============================] - 101s 14s/step
Epoch 7 Loss: 0.0003

Epoch 8/50
7/7 [==============================] - 103s 15s/step
Epoch 8 Loss: 0.0003

Epoch 9/50
7/7 [==============================] - 101s 14s/step
Epoch 9 Loss: 0.0002

Epoch 10/50
7/7 [==============================] - 102s 14s/step
Epoch 10 Loss: 0.0002


NameError: name 'checkpoint_prefix' is not defined

In [26]:
from tensorflow.keras.metrics import Precision, Recall

In [27]:
test_input, test_val, y_true = test_data.as_numpy_iterator().next()

In [28]:
y_hat = siamese_model.predict([test_input, test_val])

C:\Users\abhay\AppData\Roaming\Python\Python311\site-packages\keras\src\models\functional.py:238: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['input_image']
Received: inputs=Tensor(shape=(16, 100, 100, 3))
  warnings.warn(msg)


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step


In [29]:
y_hat

array([[0.49987206],
       [0.50050664],
       [0.50094986],
       [0.5012555 ],
       [0.50014997],
       [0.500558  ],
       [0.5005634 ],
       [0.50024974],
       [0.5026342 ],
       [0.4999548 ],
       [0.50191903],
       [0.50115037],
       [0.5005517 ],
       [0.49991474],
       [0.50286555],
       [0.50039506]], dtype=float32)

In [54]:
[1 if prediction > 0.5 else 0 for prediction in y_hat]

[0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0]

In [55]:
y_true

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
      dtype=float32)

In [36]:
m = Recall()
m.update_state(y_true, y_hat)
m.result().numpy()

0.125

In [30]:
n = Precision()
n.update_state(y_true, y_hat)
n.result().numpy()

1.0

# Saving model

In [31]:
siamese_model.save("siameseMode.h5")

In [32]:
model = make_siamese_model()

model.load_weights('siamese_model.keras')

# Real time test

In [39]:
def verify(model, detection_threshold, verification_threshold):
    results = []
    for image in os.listdir(os.path.join('application_data', 'verification_images')):
        input_img = preprocess(os.path.join('application_data', 'input_image', 'input_image.jpg'))
        validation_img = preprocess(os.path.join('application_data', 'verification_images', image))

        result = model.predict(list(np.expand_dims([input_img, validation_img], axis=1)), verbose=False)
        results.append(result)

    detection = np.sum(np.array(results) > detection_threshold)

    verification = np.sum(np.array(results) > detection_threshold)
    verification = detection / len(os.listdir(os.path.join('application_data', 'verification_images')))
    verified = verification > verification_threshold

    return results, verified
        

In [40]:
cap = cv2.VideoCapture(0)

while cap.isOpened():
    ret, frame = cap.read()

    # resizing the frame to 250 x 250
    frame = frame[170:170+250, 210:210+250, :]

    cv2.imshow("Image Collection", frame)

    if cv2.waitKey(10) & 0XFF == ord('v'):
        cv2.imwrite(os.path.join('application_data', 'input_image', 'input_image.jpg'), frame)
        results, verified = verify(model, 0.9, 0.7)
        print(verified)
    
    if cv2.waitKey(10) & 0XFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

True
True
False
False
False
False
False
False
False
False
False
True
False
False
